In [2]:
from collections import Counter, defaultdict
import itertools

In [3]:
training_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
    [("Dogs", "NOUN"), ("bark", "VERB")],
    [("Cats", "NOUN"), ("meow", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
]

In [4]:
test_sentences = [
    ["The", "dog", "barks"],
    ["A", "cat", "sleeps"],
    ["The", "big", "dog", "runs"],
    ["Dogs", "bark"],
]

In [5]:
class HMMPOSTagger:
    def __init__(self):
        self.initial = {}
        self.transition = {}
        self.emission = {}
        self.tags = set()
        self.vocab = set()
        self.tag_counts = Counter()

    def train(self, training_data):
        for sentence in training_data:
            for word, tag in sentence:
                self.tags.add(tag)
                self.vocab.add(word)
        tags_list = list(self.tags)
        num_tags = len(tags_list)
        vocab_size = len(self.vocab)

        init_counts = Counter()
        num_sentences = len(training_data)

        trans_counts = defaultdict(Counter)
        trans_source_counts = Counter()

        emit_counts = defaultdict(Counter)
        tag_counts = Counter()

        for sentence in training_data:
            first_tag = sentence[0][1]
            init_counts[first_tag] += 1
            for (word, tag) in sentence:
                emit_counts[tag][word] += 1
                tag_counts[tag] += 1
            tags_seq = [tag for _, tag in sentence]
            for t1, t2 in zip(tags_seq[:-1], tags_seq[1:]):
                trans_counts[t1][t2] += 1
                trans_source_counts[t1] += 1

        self.initial = {}
        for tag in tags_list:
            self.initial[tag] = (init_counts[tag] + 1) / (num_sentences + num_tags)

        self.transition = {tag: {} for tag in tags_list}
        for t1 in tags_list:
            denom = trans_source_counts[t1] + num_tags
            for t2 in tags_list:
                self.transition[t1][t2] = (trans_counts[t1][t2] + 1) / denom

        self.emission = {tag: {} for tag in tags_list}
        for tag in tags_list:
            denom = tag_counts[tag] + vocab_size
            for word in self.vocab:
                self.emission[tag][word] = (emit_counts[tag][word] + 1) / denom

        self.tag_counts = tag_counts

    def brute_force_decode(self, sentence):
        tags_list = list(self.tags)
        n = len(sentence)
        best_seq = None
        best_prob = -1.0

        for seq in itertools.product(tags_list, repeat=n):
            p = self.initial[seq[0]] * self.emission[seq[0]].get(sentence[0], 1e-12)
            for i in range(1, n):
                t_prev = seq[i - 1]
                t_curr = seq[i]
                p *= self.transition[t_prev][t_curr]
                p *= self.emission[t_curr].get(sentence[i], 1e-12)
            if p > best_prob:
                best_prob = p
                best_seq = seq

        return list(best_seq)



In [6]:
tagger = HMMPOSTagger()
tagger.train(training_data)


In [7]:
for sent in test_sentences:
    print(sent, "->", tagger.brute_force_decode(sent))


['The', 'dog', 'barks'] -> ['DET', 'NOUN', 'VERB']
['A', 'cat', 'sleeps'] -> ['DET', 'NOUN', 'VERB']
['The', 'big', 'dog', 'runs'] -> ['DET', 'ADJ', 'NOUN', 'VERB']
['Dogs', 'bark'] -> ['NOUN', 'VERB']
